In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
filepath="D:/spark/datasets/sf-fire-calls.csv"

In [4]:
def create_SparkSession():
    spark = SparkSession.builder.appName("sf-firecalls").getOrCreate()
    return spark

In [5]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    df1 = df.select('CallType', 'CallDate', 'City', 'Zipcode', 'Neighborhood', 'Delay')
    return df1

In [6]:
def clean_data(df):
    df1 = df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2 = df1.withColumn('Year', year(col('Date')))\
    .withColumn('Month', month(col('Date')))\
    .withColumn('Week', weekofyear(col('Date')))
    return df2

In [7]:
spark = create_SparkSession()
df = create_dataframe(spark, filepath)
df = clean_data(df)

In [8]:
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [67]:
def mapseason(data):
    if 2<data<6:
        return 'Spring'
    elif 5<data<9:
        return 'Summer'
    elif 8<data<12:
        return 'Autumn'
    else:
        return 'Winter'

In [68]:
seasonUDF = udf(mapseason, StringType())
clean_df = df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [19]:
clean_df.groupBy('Year').count().show()
    

+----+-----+
|Year|count|
+----+-----+
|2003| 8499|
|2005| 8282|
|2002| 8090|
|2006| 8174|
|2009| 8789|
|2008| 8869|
|2012| 9674|
|2010| 9341|
|2011| 9735|
|2013|10020|
|2014|10775|
|2015|11458|
|2016|11609|
|2018|10136|
|2017|12135|
|2007| 8255|
|2004| 8283|
|2000| 5459|
|2001| 7713|
+----+-----+



In [10]:
calls_2018_1 = df.filter(df['Year']==2018).select('CallType').distinct()

In [11]:
calls_2018_1.show()

+--------------------+
|            CallType|
+--------------------+
|Elevator / Escala...|
|              Alarms|
|Odor (Strange / U...|
|Citizen Assist / ...|
|              HazMat|
|        Vehicle Fire|
|               Other|
|        Outside Fire|
|   Traffic Collision|
|       Assist Police|
|Gas Leak (Natural...|
|        Water Rescue|
|   Electrical Hazard|
|      Structure Fire|
|    Medical Incident|
|          Fuel Spill|
|Smoke Investigati...|
|Train / Rail Inci...|
|           Explosion|
|  Suspicious Package|
+--------------------+



In [12]:
calls_2018_2 = df.filter(df['Year']==2018)\
.groupBy('Week')\
.count()

In [13]:
calls_2018_2.show()

+----+-----+
|Week|count|
+----+-----+
|  44|  244|
|   6|  225|
|   3|  224|
|   5|  236|
|   9|  228|
|   4|  202|
|   8|  232|
|  39|  224|
|   7|  228|
|  10|  232|
|  45|   64|
|  38|  226|
|  11|  240|
|  36|  203|
|  12|  221|
|  13|  243|
|  16|  228|
|  40|  255|
|  20|  225|
|  19|  233|
+----+-----+
only showing top 20 rows



In [17]:
df.select('Week').where(col('Year')==2018)\
.groupBy('Week')\
.count()\
.orderBy('count', ascending=False)\
.collect()[0]

Row(Week=22, count=259)

In [18]:
max_m=df.select('Week').where(col('Year')==2018).groupBy('Week').count()

In [22]:
max_m.select('Week', 'count').filter(col('count') == max_m.agg({'count':'max'}).collect()[0][0]).collect()[0][0]

22

In [45]:
#4 q
df.groupBy(col('Year'), col('Month')).count().orderBy('Year', 'Month').show(truncate=False)

+----+-----+-----+
|Year|Month|count|
+----+-----+-----+
|2000|4    |335  |
|2000|5    |680  |
|2000|6    |585  |
|2000|7    |668  |
|2000|8    |678  |
|2000|9    |655  |
|2000|10   |620  |
|2000|11   |595  |
|2000|12   |643  |
|2001|1    |622  |
|2001|2    |613  |
|2001|3    |692  |
|2001|4    |636  |
|2001|5    |682  |
|2001|6    |672  |
|2001|7    |646  |
|2001|8    |660  |
|2001|9    |577  |
|2001|10   |673  |
|2001|11   |619  |
+----+-----+-----+
only showing top 20 rows



In [66]:
#5
df.select( 'month', 'calltype').filter(col('Year')==2018).groupBy('month', 'Calltype').count().orderBy( 'month', col('count').desc()).show()

+-----+--------------------+-----+
|month|            Calltype|count|
+-----+--------------------+-----+
|    1|    Medical Incident|  692|
|    1|              Alarms|  122|
|    1|      Structure Fire|   91|
|    1|   Traffic Collision|   42|
|    1|Citizen Assist / ...|   15|
|    1|        Outside Fire|   14|
|    1|Gas Leak (Natural...|    5|
|    1|        Water Rescue|    4|
|    1|        Vehicle Fire|    4|
|    1|               Other|    3|
|    1|   Electrical Hazard|    3|
|    1|Smoke Investigati...|    3|
|    1|Elevator / Escala...|    3|
|    1|Train / Rail Inci...|    2|
|    1|Odor (Strange / U...|    2|
|    1|          Fuel Spill|    1|
|    1|              HazMat|    1|
|    2|    Medical Incident|  635|
|    2|              Alarms|  102|
|    2|      Structure Fire|   91|
+-----+--------------------+-----+
only showing top 20 rows



In [77]:
# 6 Give top five fire call types for every season of selected year (seasons are like Spring, summer, fall winter etc). 
max_s = clean_df.select('season', 'calltype').filter(col('year')==2017).groupBy('season', 'calltype').count()
max_s.select('season', 'calltype', 'count').filter(col('count')==max_s.agg({'count':'max'}).collect()[0][0]).collect()[0][0]
#Doesnt work, trying windows approach

'Spring'

In [81]:
from pyspark.sql import Window

In [85]:
max_s = clean_df.filter(col('year')==2017).groupBy('season', 'calltype').count()
window = Window.partitionBy(col('season')).orderBy(col('count').desc())
res = max_s.withColumn('rank', row_number().over(window)).filter(col('rank')<=5)

In [86]:
res.show()

+------+--------------------+-----+----+
|season|            calltype|count|rank|
+------+--------------------+-----+----+
|Autumn|    Medical Incident| 2089|   1|
|Autumn|              Alarms|  352|   2|
|Autumn|      Structure Fire|  249|   3|
|Autumn|   Traffic Collision|  135|   4|
|Autumn|        Outside Fire|   46|   5|
|Spring|    Medical Incident| 2104|   1|
|Spring|              Alarms|  354|   2|
|Spring|      Structure Fire|  276|   3|
|Spring|   Traffic Collision|  135|   4|
|Spring|               Other|   54|   5|
|Summer|    Medical Incident| 2056|   1|
|Summer|              Alarms|  337|   2|
|Summer|      Structure Fire|  239|   3|
|Summer|   Traffic Collision|  137|   4|
|Summer|Citizen Assist / ...|   43|   5|
|Winter|    Medical Incident| 2081|   1|
|Winter|              Alarms|  371|   2|
|Winter|      Structure Fire|  260|   3|
|Winter|   Traffic Collision|  124|   4|
|Winter|               Other|   54|   5|
+------+--------------------+-----+----+



In [87]:
#7 Whether fire type calls are seasonal? 
# Fire Call Types are not seasonal

In [100]:
# 8. What months within the year 2018 saw the highest number of fire calls? 

max_mm = clean_df.filter(col('year')==2018).groupBy('month').count()
w = Window.orderBy(col('count').desc())
res = max_mm.withColumn('rank', row_number().over(w)).filter(col('rank')<=5)

res1 = clean_df.filter(col('year')==2018).groupBy('month').count().orderBy(col('count').desc()).limit(5)

In [96]:
res.show()

+-----+-----+----+
|month|count|rank|
+-----+-----+----+
|   10| 1068|   1|
|    5| 1047|   2|
|    3| 1029|   3|
|    8| 1021|   4|
|    1| 1007|   5|
+-----+-----+----+



In [101]:
res1.show()

+-----+-----+
|month|count|
+-----+-----+
|   10| 1068|
|    5| 1047|
|    3| 1029|
|    8| 1021|
|    1| 1007|
+-----+-----+



In [108]:
# 9. Find which type of fire call is major calltype in each year 

max_yc = clean_df.groupBy('year', 'calltype').count()
window = Window.partitionBy(col('year')).orderBy(col('count').desc())
res = max_yc.withColumn('rank', row_number().over(window)).filter(col('rank')<2).drop('rank')

In [109]:
res.show()

+----+----------------+-----+
|year|        calltype|count|
+----+----------------+-----+
|2000|Medical Incident| 3408|
|2001|Medical Incident| 4653|
|2002|Medical Incident| 5046|
|2003|Medical Incident| 5056|
|2004|Medical Incident| 5137|
|2005|Medical Incident| 5084|
|2006|Medical Incident| 5027|
|2007|Medical Incident| 5114|
|2008|Medical Incident| 5692|
|2009|Medical Incident| 5671|
|2010|Medical Incident| 6186|
|2011|Medical Incident| 6413|
|2012|Medical Incident| 6296|
|2013|Medical Incident| 6690|
|2014|Medical Incident| 7176|
|2015|Medical Incident| 7812|
|2016|Medical Incident| 7999|
|2017|Medical Incident| 8330|
|2018|Medical Incident| 7004|
+----+----------------+-----+



In [114]:
# 10. Find out average delay in response for each call type 

clean_df.createOrReplaceTempView('Fires')
spark.sql('select calltype, avg(delay) from Fires group by calltype').show()

+--------------------+------------------+
|            calltype|        avg(delay)|
+--------------------+------------------+
|Elevator / Escala...| 4.337821933487859|
|  Aircraft Emergency|3.7731481500000004|
|              Alarms| 3.542729054508399|
|Odor (Strange / U...|       4.947959182|
|Citizen Assist / ...| 5.473342576604596|
|              HazMat| 7.527016126612904|
|           Explosion| 4.110674168539325|
|           Oil Spill| 4.977777761904762|
|        Vehicle Fire| 3.903922713407494|
|  Suspicious Package|        6.57666672|
|Extrication / Ent...| 4.391666678571428|
|               Other| 5.505155432421977|
|        Outside Fire| 4.181948425367717|
|   Traffic Collision|3.7891320888732363|
|       Assist Police|26.981903994285716|
|Gas Leak (Natural...| 4.583398778403141|
|        Water Rescue| 5.507748342145695|
|   Electrical Hazard| 5.178112038174275|
|   High Angle Rescue| 6.048958375000001|
|      Structure Fire| 3.679561015471934|
+--------------------+------------

In [115]:
# 11. Find which calltype has maximum average delay time.

spark.sql('select calltype, max(delay) from Fires group by calltype').show()

+--------------------+----------+
|            calltype|max(delay)|
+--------------------+----------+
|Elevator / Escala...| 48.716667|
|  Aircraft Emergency| 13.166667|
|              Alarms|   1844.55|
|Odor (Strange / U...| 58.116665|
|Citizen Assist / ...|     142.8|
|              HazMat|  84.76667|
|           Explosion| 22.833334|
|           Oil Spill| 29.916666|
|        Vehicle Fire| 62.833332|
|  Suspicious Package| 14.483334|
|Extrication / Ent...|      9.35|
|               Other|     405.7|
|        Outside Fire| 340.11667|
|   Traffic Collision|    239.95|
|       Assist Police| 628.61664|
|Gas Leak (Natural...| 76.566666|
|        Water Rescue|     82.85|
|   Electrical Hazard| 159.66667|
|   High Angle Rescue| 25.616667|
|      Structure Fire| 1370.2333|
+--------------------+----------+
only showing top 20 rows



In [127]:
# 12. Which neighborhood in San Francisco generated the most fire calls in 2018?

spark.sql("select Neighborhood, count(calltype) from Fires where City='SF' group by Neighborhood order by count(calltype) DESC").show()

+--------------------+---------------+
|        Neighborhood|count(calltype)|
+--------------------+---------------+
|          Tenderloin|          15482|
|     South of Market|          11098|
|             Mission|          11058|
|Financial Distric...|           7824|
|Bayview Hunters P...|           6804|
|     Sunset/Parkside|           5006|
|    Western Addition|           4698|
|            Nob Hill|           4026|
|      Outer Richmond|           3230|
|        Hayes Valley|           2980|
|  West of Twin Peaks|           2863|
| Castro/Upper Market|           2758|
|           Chinatown|           2667|
|         North Beach|           2565|
|           Excelsior|           2538|
|     Pacific Heights|           2471|
|      Bernal Heights|           2401|
|              Marina|           2362|
|        Potrero Hill|           2287|
|        Inner Sunset|           2117|
+--------------------+---------------+
only showing top 20 rows



In [128]:
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [160]:
# 13. Which neighborhoods had the worst response times to fire calls in 2018?

worst_n = clean_df.groupBy('Neighborhood').agg(avg('Delay').alias('Avg_delay')).orderBy(col('Avg_delay').desc()).limit(10)

In [161]:
worst_n.show()

+--------------------+------------------+
|        Neighborhood|         Avg_delay|
+--------------------+------------------+
|     Treasure Island| 5.471499992963636|
|            Presidio|4.9653753566522525|
|         Mission Bay|  4.53076057945946|
|        McLaren Park|  4.30982286410628|
|                None| 4.307180866893617|
|          Twin Peaks| 4.294008406036822|
|    Golden Gate Park| 4.249903661308286|
|           Lakeshore| 4.201812142693755|
|Bayview Hunters P...| 4.150424641382073|
|            Seacliff| 4.137820518730768|
+--------------------+------------------+



In [194]:
# 14. Find out calltype whose average response delay time is maximum, increases, decreases or has no relation over years.

# for each call type, for each year, find out the avg delay. Then find out if the trend is positive negetive or neautral

# max_yc = clean_df.groupBy('year', 'calltype').count()
# window = Window.partitionBy(col('year')).orderBy(col('count').desc())
# res = max_yc.withColumn('rank', row_number().over(window)).filter(col('rank')<2).drop('rank')

df_14 = clean_df.groupBy('calltype','year','delay').count()
w = Window.partitionBy('calltype', 'year').orderBy(col('delay').desc())
res = df_14.withColumn('rank', row_number().over(w)).filter(col('rank')==1).drop('rank').drop('count')
# agg(avg('Delay').alias('avg_delay')).

In [195]:
res.show()

+------------------+----+---------+
|          calltype|year|    delay|
+------------------+----+---------+
|    Administrative|2005|31.983334|
|    Administrative|2006|      1.8|
|    Administrative|2017|      3.0|
|Aircraft Emergency|2000|     6.45|
|Aircraft Emergency|2001|3.3333333|
|Aircraft Emergency|2002|      5.8|
|Aircraft Emergency|2003|13.166667|
|Aircraft Emergency|2004|3.6833334|
|Aircraft Emergency|2005|6.0833335|
|Aircraft Emergency|2006|4.2166667|
|Aircraft Emergency|2007| 5.733333|
|Aircraft Emergency|2009| 4.616667|
|Aircraft Emergency|2011| 5.983333|
|Aircraft Emergency|2012|      7.7|
|Aircraft Emergency|2013|2.4333334|
|Aircraft Emergency|2014|     7.75|
|Aircraft Emergency|2015|1.1333333|
|            Alarms|2000|     47.0|
|            Alarms|2001|     19.8|
|            Alarms|2002|23.233334|
+------------------+----+---------+
only showing top 20 rows



In [205]:
# 15. For each year find out which city has more calltypes

yw = clean_df.groupBy('year', 'city').count()
w = Window.partitionBy('year').orderBy(col('count').desc())
res = yw.withColumn('rank', row_number().over(w)).filter(col('rank')==1).drop('rank').show()

+----+-------------+-----+
|year|         city|count|
+----+-------------+-----+
|2000|           SF| 5435|
|2001|           SF| 7656|
|2002|           SF| 8044|
|2003|           SF| 8441|
|2004|           SF| 8224|
|2005|           SF| 8228|
|2006|           SF| 8114|
|2007|           SF| 8190|
|2008|           SF| 8811|
|2009|           SF| 8723|
|2010|           SF| 9272|
|2011|           SF| 9647|
|2012|           SF| 9579|
|2013|           SF| 9901|
|2014|San Francisco| 7040|
|2015|San Francisco|11304|
|2016|San Francisco|11455|
|2017|San Francisco|11973|
|2018|San Francisco| 9967|
+----+-------------+-----+



In [207]:
# 16. For every year find count of calltypes  for 5 cities which has more calls. 

yw = clean_df.groupBy('year', 'city').count()
w = Window.partitionBy('year').orderBy(col('count').desc())
res = yw.withColumn('rank', row_number().over(w)).filter(col('rank')<6).drop('rank').show()

+----+----+-----+
|year|city|count|
+----+----+-----+
|2000|  SF| 5435|
|2000|  TI|    8|
|2000|  YB|    5|
|2000|  HP|    3|
|2000| SFO|    3|
|2001|  SF| 7656|
|2001|  TI|   37|
|2001|  YB|    6|
|2001| SFO|    4|
|2001|  DC|    3|
|2002|  SF| 8044|
|2002|  TI|   18|
|2002|  PR|    7|
|2002|  HP|    6|
|2002| SFO|    5|
|2003|  SF| 8441|
|2003|  TI|   33|
|2003|  PR|    9|
|2003|  DC|    7|
|2003|  HP|    5|
+----+----+-----+
only showing top 20 rows



In [208]:
# 17. Is there a correlation between neighborhood, zip code, and number of fire calls?

yw = clean_df.groupBy('year', 'neighborhood', 'zipcode').count()
w = Window.partitionBy('year').orderBy(col('count').desc())
res = yw.withColumn('rank', row_number().over(w)).filter(col('rank')<6).drop('rank').show()

+----+--------------------+-------+-----+
|year|        neighborhood|zipcode|count|
+----+--------------------+-------+-----+
|2000|          Tenderloin|  94102|  492|
|2000|     South of Market|  94103|  401|
|2000|Bayview Hunters P...|  94124|  338|
|2000|             Mission|  94110|  337|
|2000|             Mission|  94103|  163|
|2001|          Tenderloin|  94102|  735|
|2001|     South of Market|  94103|  515|
|2001|             Mission|  94110|  514|
|2001|Bayview Hunters P...|  94124|  498|
|2001|             Mission|  94103|  233|
|2002|          Tenderloin|  94102|  713|
|2002|     South of Market|  94103|  575|
|2002|             Mission|  94110|  534|
|2002|Bayview Hunters P...|  94124|  463|
|2002|             Mission|  94103|  250|
|2003|          Tenderloin|  94102|  703|
|2003|     South of Market|  94103|  522|
|2003|             Mission|  94110|  511|
|2003|Bayview Hunters P...|  94124|  494|
|2003|             Mission|  94103|  250|
+----+--------------------+-------